<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/06_capstone_rf_vs_sjepa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Compare models with five source-grouped test folds

This notebook runs the full-data experiment from fresh model weights in each fold. Every usable clip receives exactly one out-of-fold test prediction from a model that did not train or select its settings on that source. The older g1 results belong to a different dataset and are not loaded here.

We compare Random Forest, a validation-selected S-JEPA linear probe, two fixed linear controls (visibility and mean pose), and the most common training label. All systems use identical training and test clips.

In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data-full'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints-full'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Keep each source video together

We use the same frozen five-fold registry in notebooks 02–06. A source video may have several clips; each clip may yield overlapping windows. All of those relatives stay together. Each round uses about 60% of sources for training, 20% for validation, and 20% for testing. Only notebook 06 evaluates test clips.

The splitter runs on one row per source, with condition labels used to balance source counts. It never splits windows. The loader checks the full cache, reviewed exclusions, and registry checksum. A changed cache requires a new registry and new checkpoints. See [the full method](docs/11-full-data-splits.md).

In [ ]:
from IPython.display import display
import pandas as pd
from sjepa.splits import load_full_registry, partition_records, split_summary
records, registry = load_full_registry(EXP_DIR)
FOLD = 0  # teaching example; notebook 06 independently trains all five folds
train_recs, val_recs, test_recs = partition_records(records, registry, FOLD)
display(pd.DataFrame(split_summary(records, registry)))
print('usable clips:', len(records), '| excluded raw clips:', len(registry['inventory']['exclusions']))
print('registry:', registry['registry_sha256'])

## Fix the procedure before testing

For each outer fold: train S-JEPA for 800 updates, fit a training-only probe, and score validation. Continue for 400 updates and repeat. Choose by validation source-weighted macro-F1, with ties going to the original stage. Then evaluate the chosen model on test clips. We do not refit on validation. RF uses 100 trees, depth 5, and balanced classes. Both controls use the same fixed linear-head settings as S-JEPA.

The inner validation set is the first split of a four-fold splitter within the outer development sources. We use one inner holdout, not all four inner folds. Smoke mode still checks all five outer folds, but uses 4+2 updates and a tiny model; its scores are execution checks. A normal run can take substantially longer.

In [ ]:
from sjepa.config import get_config
from sjepa.models import pick_device
from sjepa.full_experiment import run_cross_validation, new_evaluation_dir
cfg = get_config(); device = pick_device()
SMOKE = cfg.profile.endswith('smoke')
UPDATES, MORE = (4, 2) if SMOKE else (800, 400)
OUTPUT_DIR = new_evaluation_dir(EXP_DIR, registry, cfg)
print('output:', OUTPUT_DIR, '| smoke execution check:', SMOKE)
results = run_cross_validation(records, registry, cfg, device, UPDATES, MORE, OUTPUT_DIR)

## Read both ways of weighting the test predictions

Macro-F1 gives the three conditions equal importance. Clip-weighted scoring gives each clip one vote. Source-weighted scoring gives each source a total weight of one, shared among its clips. This prevents a source with 13 clips from counting 13 times as much as a source with one clip. It still scores clip predictions; it is not a person-level diagnosis.

The pooled score uses all out-of-fold predictions. The fold mean and standard deviation describe variation across rounds; they are not a confidence interval because training sets overlap. Do not choose a control or a model using this final table.

In [ ]:
rows = []
for name, scores in results['metrics'].items():
    variation = scores['source_weighted_fold_mean_std']['macro_f1']
    rows.append({'system': name, 'pooled source macro-F1': scores['source_weighted']['macro_f1'],
                 'pooled clip macro-F1': scores['clip_weighted']['macro_f1'],
                 'fold source macro-F1 mean': variation['mean'], 'fold SD': variation['std']})
display(pd.DataFrame(rows))
print('Saved one test prediction per clip to', OUTPUT_DIR / 'oof.json')
if SMOKE: print('SMOKE CHECK ONLY: these scores do not establish model quality.')

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sjepa.splits import LABELS
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ['rf', 'sjepa']):
    cm = results['metrics'][name]['source_weighted']['confusion']
    sns.heatmap(cm, annot=True, fmt='.1f', cmap='Blues', cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(f'{name}: source-weighted OOF'); ax.set_xlabel('predicted'); ax.set_ylabel('true')
plt.tight_layout(); plt.show()

## Limits of this test

A source ID identifies a recording, not a verified participant. The same person or reposted footage may occur under different IDs. Acquisition conditions may also track the labels. Source grouping prevents known within-video overlap but does not remove those problems. There is no separate external test cohort. These are development estimates on a small, previously inspected collection.

For the audit, exact counts, exclusions, commands, and statistical references, read [docs/11-full-data-splits.md](docs/11-full-data-splits.md).